<a href="https://colab.research.google.com/github/Iffat-zarrin/HDKD/blob/main/O3_DataPreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

!python --version


Python 3.12.13


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score
import os
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.20.0
GPU Available: []


In [4]:
# Load the CSV file with labels
df = pd.read_csv('/content/drive/MyDrive/Objective3_dataset/APTOS2019/archive/train_1.csv')
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nClass distribution:")
print(df['diagnosis'].value_counts().sort_index())

Dataset shape: (2930, 2)

First 5 rows:
        id_code  diagnosis
0  1ae8c165fd53          2
1  1b329a127307          1
2  1b32e1d775ea          4
3  1b3647865779          0
4  1b398c0494d1          0

Class distribution:
diagnosis
0    1434
1     300
2     808
3     154
4     234
Name: count, dtype: int64


In [8]:
# Define paths
image_dir = '/content/drive/MyDrive/Objective3_dataset/APTOS2019/archive/train_images/train_images'


# Simple function to load images (only resizing, no other preprocessing)
def load_images(image_ids, labels, img_size=(224, 224)):
    images = []
    for img_id in image_ids:
        img_path = os.path.join(image_dir, f"{img_id}.png")
        try:
            img = Image.open(img_path)
            img = img.resize(img_size)  # Only resizing
            img = np.array(img) / 255.0  # Normalize pixel values
            images.append(img)
        except:
            print(f"Could not load: {img_path}")
            # Add a black image as fallback
            images.append(np.zeros((224, 224, 3)))
    return np.array(images)

    print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

X_val shape: (586, 224, 224, 3)
y_train shape: (2344, 5)
y_val shape: (586, 5)


In [6]:
# Get all image IDs and labels
image_ids = df['id_code'].values
labels = df['diagnosis'].values

print(f"Total images to load: {len(image_ids)}")
print("Loading images (this may take a few minutes)...")

# Load all images (simple approach with just resizing)
X = load_images(image_ids, labels)
y = tf.keras.utils.to_categorical(labels, num_classes=5)

print(f"✅ Loaded {X.shape[0]} images with shape {X.shape[1:]}")

Total images to load: 2930
Loading images (this may take a few minutes)...
✅ Loaded 2930 images with shape (224, 224, 3)


In [7]:
# Split into train (80%) and validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training set: {X_train.shape[0]} images")
print(f"Validation set: {X_val.shape[0]} images")

Training set: 2344 images
Validation set: 586 images
